## XGBoost for Time Series Forecasting
#### XGBoost with Feature Engineering for Multivariate Time-Series
- **AVAILABLE** for multivariate time-series with feature engineering
- Uses lagged features, rolling statistics, and time-based features


In [1]:
# Import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


In [2]:
'''
Load dataset and preprocessing
-> train | test | submission | prediction
'''

# train data
train_data = pd.read_csv('./dataset/train/train.csv')
# ['date'] -> datetime
train_data['date'] = pd.to_datetime(train_data['date'], format='%Y-%m-%d')
# ordinal date feature
train_data['date_ordinal'] = train_data['date'].map(datetime.toordinal)
# store_menu_id
train_data['store_menu_id'] = train_data['store'] + "_" + train_data['menu']


# test data
for i in range(0, 10):
    test = pd.read_csv(f"./dataset/test/TEST_0{i}.csv")
    test['date'] = pd.to_datetime(test['date'], format='%Y-%m-%d')
    test['date_ordinal'] = test['date'].map(datetime.toordinal)
    test['store_menu_id'] = test['store'] + "_" + test['menu']
    # test_data_{i} for all test datasets
    globals()[f'test_data_{i}'] = test

# submission format
submission = pd.read_csv("./result/sample_submission_date.csv")

# Prediction result
all_preds = []


In [3]:
# Feature Engineering Functions

def create_time_features(df):
    """Create time-based features"""
    df = df.copy()
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofyear'] = df['date'].dt.dayofyear
    df['weekofyear'] = df['date'].dt.isocalendar().week
    df['quarter'] = df['date'].dt.quarter
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    
    # Cyclical encoding for better representation
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    return df

def create_lag_features(df, target_col='sales', lags=[1, 2, 3, 7, 14, 21, 28]):
    """Create lagged features"""
    df = df.copy()
    df = df.sort_values(['store_menu_id', 'date']).reset_index(drop=True)
    
    for lag in lags:
        df[f'{target_col}_lag_{lag}'] = df.groupby('store_menu_id')[target_col].shift(lag)
    
    return df

def create_rolling_features(df, target_col='sales', windows=[3, 7, 14, 28]):
    """Create rolling window features"""
    df = df.copy()
    df = df.sort_values(['store_menu_id', 'date']).reset_index(drop=True)
    
    for window in windows:
        # Rolling mean
        df[f'{target_col}_rolling_mean_{window}'] = df.groupby('store_menu_id')[target_col].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )
        
        # Rolling std
        df[f'{target_col}_rolling_std_{window}'] = df.groupby('store_menu_id')[target_col].transform(
            lambda x: x.rolling(window=window, min_periods=1).std().fillna(0)
        )
        
        # Rolling min/max
        df[f'{target_col}_rolling_min_{window}'] = df.groupby('store_menu_id')[target_col].transform(
            lambda x: x.rolling(window=window, min_periods=1).min()
        )
        df[f'{target_col}_rolling_max_{window}'] = df.groupby('store_menu_id')[target_col].transform(
            lambda x: x.rolling(window=window, min_periods=1).max()
        )
    
    return df

def create_target_encoding(df, categorical_cols, target_col='sales'):
    """Create target encoding features"""
    df = df.copy()
    
    for col in categorical_cols:
        target_mean = df.groupby(col)[target_col].mean()
        df[f'{col}_target_mean'] = df[col].map(target_mean)
        
        target_std = df.groupby(col)[target_col].std().fillna(0)
        df[f'{col}_target_std'] = df[col].map(target_std)
    
    return df


In [4]:
def prepare_features(df):
    """Prepare all features for XGBoost"""
    df = df.copy()
    
    # Time features
    df = create_time_features(df)
    
    # Lag features
    df = create_lag_features(df)
    
    # Rolling features
    df = create_rolling_features(df)
    
    # Target encoding for categorical features
    categorical_cols = ['store', 'menu']
    df = create_target_encoding(df, categorical_cols)
    
    # Label encoding for categorical variables
    le_store = LabelEncoder()
    le_menu = LabelEncoder()
    le_store_menu = LabelEncoder()
    
    df['store_encoded'] = le_store.fit_transform(df['store'])
    df['menu_encoded'] = le_menu.fit_transform(df['menu'])
    df['store_menu_encoded'] = le_store_menu.fit_transform(df['store_menu_id'])
    
    return df, le_store, le_menu, le_store_menu


In [5]:
def predict_with_xgboost(train_df, test_df, sid):
    """Predict using XGBoost for a specific store_menu_id"""
    # Filter data for specific store_menu_id
    train_sid = train_df[train_df['store_menu_id'] == sid].copy()
    test_sid = test_df[test_df['store_menu_id'] == sid].copy()
    
    if len(train_sid) == 0 or len(test_sid) == 0:
        raise ValueError(f"No data found for {sid}")
    
    # Combine and sort data
    combined_data = pd.concat([train_sid, test_sid], ignore_index=True)
    combined_data = combined_data.sort_values('date').reset_index(drop=True)
    
    # Prepare features
    combined_data, _, _, _ = prepare_features(combined_data)
    
    # Define feature columns (exclude target and identifier columns)
    exclude_cols = ['date', 'store', 'menu', 'store_menu_id', 'sales']
    feature_cols = [col for col in combined_data.columns if col not in exclude_cols]
    
    # Split back to train and test
    train_end_idx = len(train_sid)
    train_features = combined_data.iloc[:train_end_idx]
    test_features = combined_data.iloc[train_end_idx:]
    
    # Get last 28 days for prediction input
    input_end_ordinal = test_features['date_ordinal'].max()
    prediction_input = combined_data[
        (combined_data['date_ordinal'] <= input_end_ordinal) & 
        (combined_data['date_ordinal'] > input_end_ordinal - 28)
    ].copy()
    
    if len(prediction_input) != 28:
        raise ValueError(f"{sid} does not have exactly 28 days of input data.")
    
    # Prepare training data (use data before the test period)
    train_data_for_model = combined_data[
        combined_data['date_ordinal'] <= input_end_ordinal
    ].copy()
    
    # Remove rows with NaN values (due to lag features)
    train_data_for_model = train_data_for_model.dropna()
    
    if len(train_data_for_model) < 50:  # Minimum training samples
        # Fallback to simple mean prediction
        recent_mean = prediction_input['sales'].tail(7).mean()
        forecast = np.full(7, recent_mean if not np.isnan(recent_mean) else 0)
    else:
        # Train XGBoost model
        X_train = train_data_for_model[feature_cols]
        y_train = train_data_for_model['sales']
        
        # XGBoost parameters
        params = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'max_depth': 6,
            'learning_rate': 0.1,
            'n_estimators': 100,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': 42,
            'verbosity': 0
        }
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        
        # Predict next 7 days iteratively
        forecast = []
        current_data = combined_data.copy()
        
        for day in range(7):
            # Create next day data
            next_date = prediction_input['date'].max() + timedelta(days=day+1)
            next_ordinal = input_end_ordinal + day + 1
            
            # Create a new row for prediction
            next_row = prediction_input.iloc[-1:].copy()
            next_row['date'] = next_date
            next_row['date_ordinal'] = next_ordinal
            next_row['sales'] = np.nan  # Unknown target
            
            # Add to current data and recreate features
            temp_data = pd.concat([current_data, next_row], ignore_index=True)
            temp_data = temp_data.sort_values('date').reset_index(drop=True)
            temp_data, _, _, _ = prepare_features(temp_data)
            
            # Get the prediction row
            pred_row = temp_data.iloc[-1:]
            
            # Handle NaN values in features
            pred_features = pred_row[feature_cols].fillna(0)
            
            # Make prediction
            pred_value = model.predict(pred_features)[0]
            pred_value = max(0, pred_value)  # Ensure non-negative
            
            forecast.append(pred_value)
            
            # Update the prediction row with the predicted value
            temp_data.loc[temp_data.index[-1], 'sales'] = pred_value
            current_data = temp_data.copy()
        
        forecast = np.array(forecast)
    
    # Create forecast dates
    forecast_ordinals = np.arange(input_end_ordinal + 1, input_end_ordinal + 8)
    forecast_dates = pd.to_datetime([datetime.fromordinal(int(o)) for o in forecast_ordinals])
    
    return pd.DataFrame({
        'date': forecast_dates,
        'store_menu_id': sid,
        'sales': forecast
    })


In [6]:
def run_recursive_forecasting_xgboost(train_df, test_data_list):
    """Run recursive forecasting using XGBoost"""
    all_predictions = []
    
    for i, test_df in enumerate(test_data_list):
        test_df = test_df.copy()
        test_df['date'] = pd.to_datetime(test_df['date'])
        test_df = test_df.sort_values(['store_menu_id', 'date'])
        
        pred_list = []
        store_menu_ids = test_df['store_menu_id'].unique()
        
        for sid in tqdm(store_menu_ids, desc=f"Predicting TEST_{i} with XGBoost"):
            try:
                pred_df = predict_with_xgboost(train_df, test_df, sid)
                pred_list.append(pred_df)
                
                # Update train_df: add current test + prediction
                test_part = test_df[test_df['store_menu_id'] == sid]
                train_df = pd.concat([train_df, test_part, pred_df])
                
            except Exception as e:
                print(f"Failed for {sid}: {e}")
                # Create fallback prediction (zero or mean)
                test_part = test_df[test_df['store_menu_id'] == sid]
                if len(test_part) > 0:
                    input_end_ordinal = test_part['date_ordinal'].max()
                    forecast_ordinals = np.arange(input_end_ordinal + 1, input_end_ordinal + 8)
                    forecast_dates = pd.to_datetime([datetime.fromordinal(int(o)) for o in forecast_ordinals])
                    fallback_pred = pd.DataFrame({
                        'date': forecast_dates,
                        'store_menu_id': sid,
                        'sales': np.zeros(7)  # Fallback to zero prediction
                    })
                    pred_list.append(fallback_pred)
        
        if pred_list:
            all_predictions.append(pd.concat(pred_list))
    
    return pd.concat(all_predictions) if all_predictions else pd.DataFrame()


In [7]:
# Prepare test data list
test_data_list = []
for i in range(10):
    test_data_list.append(globals()[f'test_data_{i}'])

# Run XGBoost forecasting
print("Starting XGBoost recursive forecasting...")
final_predictions = run_recursive_forecasting_xgboost(train_data.copy(), test_data_list)

print(f"Total predictions generated: {len(final_predictions)}")
print(final_predictions.head())


Starting XGBoost recursive forecasting...


Predicting TEST_9 with XGBoost: 100%|██████████| 193/193 [01:53<00:00,  1.69it/s]

Total predictions generated: 13510
        date       store_menu_id      sales
0 2024-07-14  느티나무 셀프BBQ_1인 수저세트  11.838753
1 2024-07-15  느티나무 셀프BBQ_1인 수저세트  15.065319
2 2024-07-16  느티나무 셀프BBQ_1인 수저세트  12.578959
3 2024-07-17  느티나무 셀프BBQ_1인 수저세트  12.875420
4 2024-07-18  느티나무 셀프BBQ_1인 수저세트  12.758337


In [8]:
# Create submission file
def create_submission_file(predictions_df, submission_template, output_path):
    """Create submission file in the required format"""
    
    # Initialize submission with template
    submission_df = submission_template.copy()
    
    # Group predictions by store_menu_id and date
    predictions_pivot = predictions_df.pivot_table(
        index='date', 
        columns='store_menu_id', 
        values='sales', 
        fill_value=0
    )
    
    # Fill submission template with predictions
    for col in submission_df.columns:
        if col in predictions_pivot.columns:
            # Get predictions for this store_menu combination
            pred_values = predictions_pivot[col].values
            if len(pred_values) == len(submission_df):
                submission_df[col] = pred_values
            else:
                print(f"Warning: Mismatch in prediction length for {col}")
        else:
            print(f"Warning: No predictions found for {col}")
            submission_df[col] = 0  # Fill with zeros if no prediction
    
    # Save submission file
    submission_df.to_csv(output_path, index=False)
    print(f"Submission file saved to: {output_path}")
    
    return submission_df

# Create and save submission
if len(final_predictions) > 0:
    submission_result = create_submission_file(
        final_predictions, 
        submission, 
        "./result/xgboost_submission.csv"
    )
    
    print("\nSubmission file shape:", submission_result.shape)
    print("Sample submission values:")
    print(submission_result.iloc[:5, :5])
else:
    print("No predictions generated. Please check the model.")


Submission file saved to: ./result/xgboost_submission.csv

Submission file shape: (70, 194)
Sample submission values:
   date  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ_BBQ55(단체)  느티나무 셀프BBQ_대여료 30,000원  \
0     0           11.838753                   0.0               10.948333   
1     0           15.065319                   0.0                9.717954   
2     0           12.578959                   0.0               10.106912   
3     0           12.875420                   0.0                9.059958   
4     0           12.758337                   0.0                8.266079   

   느티나무 셀프BBQ_대여료 60,000원  
0                6.285997  
1                6.527453  
2                6.544076  
3                6.477731  
4                4.492177  


In [9]:
# Model evaluation and feature importance (optional)
def evaluate_model_performance(predictions_df):
    """Basic evaluation of model performance"""
    
    print("=== Model Performance Summary ===")
    print(f"Total predictions: {len(predictions_df)}")
    print(f"Unique store-menu combinations: {predictions_df['store_menu_id'].nunique()}")
    print(f"Prediction date range: {predictions_df['date'].min()} to {predictions_df['date'].max()}")
    
    print("\n=== Sales Prediction Statistics ===")
    print(predictions_df['sales'].describe())
    
    print("\n=== Predictions by Store-Menu (Top 10) ===")
    top_predictions = predictions_df.groupby('store_menu_id')['sales'].sum().sort_values(ascending=False).head(10)
    print(top_predictions)

if len(final_predictions) > 0:
    evaluate_model_performance(final_predictions)


=== Model Performance Summary ===
Total predictions: 13510
Unique store-menu combinations: 193
Prediction date range: 2024-07-14 00:00:00 to 2025-05-31 00:00:00

=== Sales Prediction Statistics ===
count    13510.000000
mean         8.949653
std         34.115774
min          0.000000
25%          0.013807
50%          1.223112
75%          5.079653
max        672.623352
Name: sales, dtype: float64

=== Predictions by Store-Menu (Top 10) ===
store_menu_id
화담숲주막_해물파전         8766.130263
포레스트릿_꼬치어묵         7719.280804
포레스트릿_떡볶이          5291.857719
포레스트릿_생수           4597.459271
카페테리아_수제 등심 돈까스    3709.010194
화담숲카페_아메리카노 ICE    3194.848863
포레스트릿_치즈 핫도그       3055.709643
담하_공깃밥             2847.544447
미라시아_브런치(대인) 주말    2620.990431
화담숲주막_느린마을 막걸리     2340.681410
Name: sales, dtype: float64
